<a href="https://colab.research.google.com/github/aounraza379/SecureMed-Fed/blob/main/securemed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random

# Set the number of data points (simulating about 15-16 minutes of data)
data_points = 1000

# Normal heart rates (60 to 90 beats per minute)
heart_rates = [random.randint(60, 90) for _ in range(data_points)]

# Creating a "Cardiac Anomaly" (Emergency)
# Changing the last 50 readings to be dangerously high
for i in range(950, 1000):
    heart_rates[i] = random.randint(150, 180)

# Timestamp for each reading
timestamps = pd.date_range(start="2026-04-01 08:00:00", periods=data_points, freq='S')

# Spreadsheet (DataFrame)
df = pd.DataFrame({
    'Timestamp': timestamps,
    'Heart_Rate': heart_rates
})

# Save it as a CSV file
df.to_csv('patient_data.csv', index=False)

print("File 'patient_data.csv' has been created.")
print(df.tail(10))

In [ ]:
# Load csv
data = pd.read_csv('patient_data.csv')

# Safety Threshold (120 BPM is the danger zone for an elderly person at rest)
THRESHOLD = 120
alert_triggered = False

print("--- SecureMed Watchdog Scanning Started ---")

# Scan 'Heart_Rate' column
for index, row in data.iterrows():
    current_bpm = row['Heart_Rate']
    timestamp = row['Timestamp']

    if current_bpm > THRESHOLD:
        print(f"ALERT! High Heart Rate Detected: {current_bpm} BPM at {timestamp}")
        alert_triggered = True

if alert_triggered:
    print("\n--- FINAL STATUS: CARDIAC ANOMALY DETECTED. EMERGENCY SERVICES NOTIFIED. ---")
else:
    print("\n--- FINAL STATUS: Patient is Stable. ---")

In [ ]:
import numpy as np
import hashlib

def generate_secure_update(heart_rate_column):
    """
    Simulates a Local Differential Privacy update for Federated Learning.
    Masks raw data before transmission to the central server.
    """
    # Calculate local statistics
    local_mean = heart_rate_column.mean()
    local_std = heart_rate_column.std()

    # Add Gaussian Noise (Differential Privacy) to mask the exact mean
    noise = np.random.normal(0, 0.1, 1)[0]
    masked_update = local_mean + noise

    # Generate a unique hash for the device to ensure data integrity (ZKP Foundation)
    device_id = "Device_Elderly_001"
    data_hash = hashlib.sha256(str(masked_update).encode()).hexdigest()

    return {
        "masked_mean": round(masked_update, 2),
        "integrity_hash": data_hash,
        "status": "Anomaly_Detected" if local_mean > 110 else "Normal"
    }

# Execute Local Privacy Masking
local_data = pd.read_csv('patient_data.csv')
secure_packet = generate_secure_update(local_data['Heart_Rate'])

print(f"--- Transmission Packet Created ---")
print(f"Masked Mean (Shared): {secure_packet['masked_mean']}")
print(f"Integrity Hash (Encrypted): {secure_packet['integrity_hash']}")
print(f"System Status: {secure_packet['status']}")

In [ ]:
import hashlib
import time

def generate_emergency_proof(heart_rate, secret_key="PATIENT_SECRET_001"):
    """
    Simulates a Zero-Knowledge Proof (ZKP) for emergency alerts.
    Verifies the severity of the event without exposing raw vitals.
    """
    timestamp = str(int(time.time()))

    # A "Commitment": A cryptographic hash of the data + a secret key
    # This acts as a 'digital seal' that cannot be altered.
    commitment = hashlib.sha256(f"{heart_rate}{secret_key}{timestamp}".encode()).hexdigest()

    # Define the 'Proof' condition (Is it a real emergency?)
    is_emergency = heart_rate > 120

    # Create the 'Proof' packet for the hospital
    # We send the commitment and the status, but NOT the secret_key or the exact heart_rate
    proof_packet = {
        "commitment": commitment,
        "is_emergency": is_emergency,
        "timestamp": timestamp,
        "verification_status": "PENDING"
    }

    return proof_packet

def verify_emergency_proof(proof_packet, original_commitment):
    """
    Simulates the server-side verification of the cryptographic seal.
    """
    if proof_packet["commitment"] == original_commitment:
        proof_packet["verification_status"] = "VERIFIED_AUTHENTIC"
    else:
        proof_packet["verification_status"] = "INVALID_PROOF"

    return proof_packet

# Execute ZKP Logic for a sample anomaly
emergency_proof = generate_emergency_proof(165)
verified_result = verify_emergency_proof(emergency_proof, emergency_proof["commitment"])

print(f"--- Zero-Knowledge Proof (ZKP) Verification ---")
print(f"Emergency Status: {verified_result['is_emergency']}")
print(f"Cryptographic Seal: {verified_result['commitment']}")
print(f"Verification Result: {verified_result['verification_status']}")

In [ ]:
import plotly.graph_objects as go
from datetime import datetime

def launch_doctor_dashboard(dataframe, alert_status, proof):
    """
    Simulates a Real-Time Medical Dashboard.
    Visualizes patient data and highlights verified emergency alerts.
    """
    # Create the base heart rate graph
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=dataframe['Timestamp'],
        y=dataframe['Heart_Rate'],
        mode='lines',
        name='Patient Heart Rate',
        line=dict(color='royalblue', width=2)
    ))

    # If a ZKP verified alert exists, highlight it on the dashboard
    if alert_status == "VERIFIED_AUTHENTIC":
        # Find the max heart rate to point the arrow at the emergency
        max_hr = dataframe['Heart_Rate'].max()
        max_time = dataframe.loc[dataframe['Heart_Rate'].idxmax(), 'Timestamp']

        fig.add_annotation(
            x=max_time,
            y=max_hr,
            text="CRITICAL ANOMALY: EMERGENCY TRIGGERED",
            showarrow=True,
            arrowhead=1,
            bgcolor="red",
            font=dict(color="white")
        )

    # Final Dashboard Layout
    fig.update_layout(
        title='SecureMed-Fed: Real-Time Elderly Monitoring (Global Node 001)',
        xaxis_title='Time (Real-Time Stream)',
        yaxis_title='Beats Per Minute (BPM)',
        template='plotly_dark'
    )

    fig.show()

# Execution: Combining all phases into the final UI
launch_doctor_dashboard(data, verified_result['verification_status'], verified_result)